In [19]:
from pathlib import Path
import re
import pandas as pd

Intra-dataset evaluation: Train on and test on the same <dataset>

In [20]:
LOG_ROOT = Path("./checkpoints")

In [21]:
filename_pattern = re.compile(r"^train_(?P<dataset>.+)\.log$")
result_pattern = re.compile(
    r"TEST\s*\|\s*"
    r"loss=(?P<loss>[-+]?\d*\.?\d+)\s+"
    r"plcc=(?P<plcc>[-+]?\d*\.?\d+)\s+"
    r"srcc=(?P<srcc>[-+]?\d*\.?\d+)\s+"
    r"rmse=(?P<rmse>[-+]?\d*\.?\d+)"
)
records = []
for log_path in LOG_ROOT.rglob("*.log"):
    filename_match = filename_pattern.match(log_path.name)
    dataset = filename_match.group("dataset")

    text = log_path.read_text(encoding="utf-8", errors="ignore")
    matches = list(result_pattern.finditer(text))
    if not matches:
        continue
    records.append({
        "dataset": dataset,
        "srcc": float(matches[-1].group("srcc")),
        "plcc": float(matches[-1].group("plcc")),
    })
train_df = (pd.DataFrame(records).sort_values("dataset", ignore_index=True)[["dataset", "srcc", "plcc"]])
train_df

,dataset,srcc,plcc
0,finevd,0.8596,0.8579
1,konvid_1k,0.7970,0.8171
2,kvq,0.8768,0.8781
3,live_vqc,0.6386,0.7192
4,live_yt_gaming,0.8554,0.8755
5,lsvq,0.8753,0.8760
6,shorts-hdr-dataset_hdr2sdr,0.5431,0.6657
7,shorts-hdr-dataset_sdr,0.7884,0.8182
8,youtube_ugc,0.8325,0.8548


Cross-dataset evaluation: Train on <trainOn> dataset and test on "testOn" dataset

In [22]:
LOG_ROOT = Path("./checkpoints_transfer")

In [23]:
filename_pattern = re.compile(r"^transfer_(?P<trainOn>.+?)_test_on_(?P<testOn>.+?)\.log$")
result_pattern = re.compile(
    r"TEST_ONLY\s*\|\s*"
    r"loss=(?P<loss>[-+]?\d*\.?\d+)\s+"
    r"plcc=(?P<plcc>[-+]?\d*\.?\d+)\s+"
    r"srcc=(?P<srcc>[-+]?\d*\.?\d+)\s+"
    r"rmse=(?P<rmse>[-+]?\d*\.?\d+)"
)
records = []
for log_path in LOG_ROOT.rglob("transfer_*_test_on_*.log"):
    filename_match = filename_pattern.match(log_path.name)
    trainOn = filename_match.group("trainOn")
    testOn = filename_match.group("testOn")

    text = log_path.read_text(encoding="utf-8", errors="ignore")
    matches = list(result_pattern.finditer(text))
    if not matches:
        continue
    records.append({
            "trainOn": trainOn,
            "testOn": testOn,
            "srcc": float(matches[-1].group("srcc")),
            "plcc": float(matches[-1].group("plcc")),
    })
transfer_df = (pd.DataFrame(records).sort_values(["trainOn", "testOn"], ignore_index=True))
transfer_df

,trainOn,testOn,srcc,plcc
0,kvq,shorts-hdr-dataset_hdr2sdr,0.4761,0.5885
1,kvq,shorts-hdr-dataset_sdr,0.7548,0.8065
2,shorts-hdr-dataset_hdr2sdr,kvq,0.5976,0.5693
3,shorts-hdr-dataset_hdr2sdr,shorts-hdr-dataset_sdr,0.6166,0.6802
4,shorts-hdr-dataset_sdr,kvq,0.7342,0.7451
5,shorts-hdr-dataset_sdr,shorts-hdr-dataset_hdr2sdr,0.5447,0.6958


Cross-dataset evaluation: Train on "trainOn" dataset and finetune on "fintuneOn" dataset

In [24]:
filename_pattern = re.compile(r"^transfer_(?P<trainOn>.+?)_finetune_on_(?P<finetuneOn>.+?)\.log$")
finetune_result_pattern = re.compile(
    r"FINETUNE TEST\s*\|\s*"
    r"loss=(?P<loss>[-+]?\d*\.?\d+)\s+"
    r"plcc=(?P<plcc>[-+]?\d*\.?\d+)\s+"
    r"srcc=(?P<srcc>[-+]?\d*\.?\d+)\s+"
    r"rmse=(?P<rmse>[-+]?\d*\.?\d+)"
)
records = []
for log_path in LOG_ROOT.rglob("transfer_*_finetune_on_*.log"):
    filename_match = filename_pattern.match(log_path.name)
    trainOn = filename_match.group("trainOn")
    finetuneOn = filename_match.group("finetuneOn")

    text = log_path.read_text(encoding="utf-8", errors="ignore")
    matches = list(finetune_result_pattern.finditer(text))
    if not matches:
        continue
    records.append({
            "trainOn": trainOn,
            "finetuneOn": finetuneOn,
            "srcc": float(matches[-1].group("srcc")),
            "plcc": float(matches[-1].group("plcc")),
    })
finetune_df = (pd.DataFrame(records).sort_values(["trainOn", "finetuneOn"], ignore_index=True))
finetune_df

,trainOn,finetuneOn,srcc,plcc
0,kvq,shorts-hdr-dataset_hdr2sdr,0.6412,0.7225
1,kvq,shorts-hdr-dataset_sdr,0.8291,0.8683
2,shorts-hdr-dataset_hdr2sdr,kvq,0.8743,0.8792
3,shorts-hdr-dataset_hdr2sdr,shorts-hdr-dataset_sdr,0.8183,0.8561
4,shorts-hdr-dataset_sdr,kvq,0.8862,0.8883
5,shorts-hdr-dataset_sdr,shorts-hdr-dataset_hdr2sdr,0.6589,0.8013


Complexity_test

In [25]:
complexity_csv_path = "../test_videos/complexity_test/"

In [26]:
resolution_df1 = pd.read_csv(f"{complexity_csv_path}complexity_resolution_KVQ_0546.csv").assign(video="KVQ_0546")
resolution_df2 = pd.read_csv(f"{complexity_csv_path}complexity_resolution_SDR_Animal_5ngj.csv").assign(video="SDR_Animal_5ngj")
resolution_df = pd.concat([resolution_df1, resolution_df2], ignore_index=True)
resolution_df

,model,540P,720P,1080P,2160P,MOS,video
0,FAST-VQA,0.638,0.745,1.020,2.680,4.017,KVQ_0546
1,FasterVQA,0.428,0.480,0.573,1.077,4.356,KVQ_0546
2,DOVER,1.018,1.163,1.511,3.662,4.223,KVQ_0546
3,Our,0.364,0.485,0.753,2.437,3.650,KVQ_0546
4,FAST-VQA,0.599,0.673,0.909,2.217,3.319,SDR_Animal_5ngj
5,FasterVQA,0.489,0.547,0.696,1.343,3.556,SDR_Animal_5ngj
6,DOVER,0.920,1.022,1.293,2.783,3.814,SDR_Animal_5ngj
7,Our,0.313,0.405,0.697,2.137,3.878,SDR_Animal_5ngj


In [27]:
complexity_df = pd.read_csv(f"{complexity_csv_path}complexity_test.csv")
complexity_df

,model,video,Time,Predicted_MOS,Scaled_MOS,MOS
0,FAST-VQA,SDR_Gameplay_wcq2,0.822,0.248790,1.995,4.156
1,FAST-VQA,SDR_Gameplay_s0pc,0.820,0.550190,3.201,4.264
2,FAST-VQA,SDR_Gameplay_z5fz,0.823,0.538800,3.155,4.221
3,FAST-VQA,SDR_Animal_5ngj,0.846,0.579850,3.319,4.308
4,FAST-VQA,0546,0.895,0.754290,4.017,2.389
5,FasterVQA,SDR_Gameplay_wcq2,0.502,0.299890,2.200,4.156
6,FasterVQA,SDR_Gameplay_s0pc,0.524,0.706330,3.825,4.264
7,FasterVQA,SDR_Gameplay_z5fz,0.499,0.400900,2.604,4.221
8,FasterVQA,SDR_Animal_5ngj,0.500,0.638990,3.556,4.308
9,FasterVQA,0546,0.484,0.839070,4.356,2.389
